ResNet 모델을 불러와 새로운 이미지 데이터셋을 분류하세요.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models

In [2]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train=torch.tensor(x_train, dtype=torch.float32).unsqueeze(1) / 255.0
x_test=torch.tensor(x_test, dtype=torch.float32).unsqueeze(1) / 255.0
x_train=x_train.repeat(1,3,1,1)
x_test=x_test.repeat(1,3,1,1)
y_train=torch.tensor(y_train, dtype=torch.long)
y_test=torch.tensor(y_test, dtype=torch.long)
train_dataset=TensorDataset(x_train, y_train)
test_dataset=TensorDataset(x_test, y_test)
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [3]:
base_model = models.resnet50(weights=None)
base_model = nn.Sequential(*list(base_model.children())[:-2])

In [4]:
num_classes=10
class CustomResNet50(nn.Module):
  def __init__(self, num_classes):
    super(CustomResNet50, self).__init__()
    self.base_model=base_model
    self.global_avg_pool=nn.AdaptiveAvgPool2d((1,1))
    self.fc1=nn.Linear(2048,256)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(256,num_classes)
  def forward(self, x):
    x=self.base_model(x)
    x=self.global_avg_pool(x)
    x=torch.flatten(x,1)
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    return x
model=CustomResNet50(num_classes=num_classes)

In [5]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(), lr=0.0001)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")  # cuda 확인용

model = CustomResNet50(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

사용 중인 디바이스: cuda
Epoch 1, Loss: 0.550419338569045
Epoch 2, Loss: 0.15250332434525093
Epoch 3, Loss: 0.0939844772182405
Epoch 4, Loss: 0.06897143243669222
Epoch 5, Loss: 0.05576914429034417
Epoch 6, Loss: 0.046245946886219705
Epoch 7, Loss: 0.03751272205067022
Epoch 8, Loss: 0.032687955668677264
Epoch 9, Loss: 0.02799770538128214
Epoch 10, Loss: 0.0237362901951458


In [7]:
model.eval()
correct=0
total=0
with torch.no_grad():
  for inputs, labels in test_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outputs=model(inputs)
    _, predicted=torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()
accuracy=correct/total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 98.91%


이미지 데이터셋과 사전 훈련된 VGG16 모델을 가져와 전이 학습을 수행하세요.


In [8]:
base_model=models.vgg16(weights=None)
base_model.classifier=nn.Sequential(*list(base_model.classifier.children())[:-1])

In [9]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1) / 255.0
x_test  = torch.tensor(x_test,  dtype=torch.float32).unsqueeze(1) / 255.0
x_train = torch.nn.functional.interpolate(x_train, size=(32, 32), mode='bilinear', align_corners=False)
x_test  = torch.nn.functional.interpolate(x_test,  size=(32, 32), mode='bilinear', align_corners=False)

x_train = x_train.repeat(1, 3, 1, 1)
x_test  = x_test.repeat(1, 3, 1, 1)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)
train_dataset = TensorDataset(x_train, y_train)
test_dataset  = TensorDataset(x_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [10]:
class CustomVGG16(nn.Module):
  def __init__(self, num_classes):
    super(CustomVGG16, self).__init__()
    self.base_model=base_model
    self.global_avg_pool=nn.AdaptiveAvgPool2d((1,1))
    self.fc1=nn.Linear(512,256)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(256,num_classes)
    self.softmax=nn.Softmax(dim=1)
  def forward(self, x):
    x=self.base_model.features(x)
    x=self.global_avg_pool(x)
    x=torch.flatten(x,1)
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    return x
model=CustomVGG16(num_classes=num_classes)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")  # cuda 확인용

model = CustomVGG16(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

사용 중인 디바이스: cuda
Epoch 1, Loss: 0.18273407212967674
Epoch 2, Loss: 0.056115643184127595
Epoch 3, Loss: 0.04246578771797504
Epoch 4, Loss: 0.034336801482209314
Epoch 5, Loss: 0.028209254225287684
Epoch 6, Loss: 0.023724801541330574
Epoch 7, Loss: 0.02119306260539692
Epoch 8, Loss: 0.01698465053304608
Epoch 9, Loss: 0.017890680970190042
Epoch 10, Loss: 0.013119883661399963


 동일한 데이터셋에서 ResNet과 VGG16을 각각 학습시켜 성능을 비교하세요.

In [12]:
model.eval()
correct=0
total=0
with torch.no_grad():
  for inputs, labels in test_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outputs=model(inputs)
    _, predicted=torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()
accuracy=correct/total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 99.50%


## 같은 MNIST 데이터에서
## RSNET50 정확도: 98.92%
## VGG16 정확도 : 99.50%

가상 데이터셋을 생성한 뒤, GridSearch와 RandomSearch 기법으로 하이퍼파라미터 튜닝을 진행하세요.

## Grid Search

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
num_classes=10
input_shape=(32,32,3)
num_samples=1000
X=np.random.rand(num_samples, *input_shape)
y=np.random.randint(num_classes, size=num_samples)
X,y=shuffle(X,y)
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2, random_state=42)

In [14]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
def create_model(learning_rate=0.001):
  base_model=ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
  x=base_model.output
  x=GlobalAveragePooling2D()(x)
  x=Dense(1024, activation="relu")(x)
  predictions=Dense(num_classes, activation="softmax")(x)
  model=Model(inputs=base_model.input, outputs=predictions)
  opt=Adam(learning_rate=learning_rate)
  model.compile(optimizer=opt, loss="sparse_categorical_crossentropy",metrics=['accuracy'])
  return model

In [15]:
learning_rates=[0.001,0.01,0.1]
batch_sizes=[16,32,64]
epochs=10
best_accuracy=10
best_accuracy=0
best_param={}
for lr in learning_rates:
  for batch_size in batch_sizes:
    model=create_model(learning_rate=lr)
    model.fit(X_train, y_train, batch_size=batch_size, validation_split=0.2, verbose=1)
    loss, accuracy=model.evaluate(X_test, y_test, verbose=0)
    print(f"Learning Rate: {lr}, Batch Size: {batch_size}, Test Accuracy: {accuracy}")
    if accuracy>best_accuracy:
      best_accuracy=accuracy
      best_params={"learning_rate": lr, "batch_size":batch_size}
print("Best Parameters:", best_params)
print("Best Test Accuracy:", best_accuracy)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 59s 137ms/step - accuracy: 0.0984 - loss: 3.4708 - val_accuracy: 0.0938 - val_loss: 2038.9124
Learning Rate: 0.001, Batch Size: 16, Test Accuracy: 0.125
20/20 ━━━━━━━━━━━━━━━━━━━━ 57s 260ms/step - accuracy: 0.0844 - loss: 3.4537 - val_accuracy: 0.0812 - val_loss: 43.9935
Learning Rate: 0.001, Batch Size: 32, Test Accuracy: 0.10999999940395355
10/10 ━━━━━━━━━━━━━━━━━━━━ 63s 852ms/step - accuracy: 0.1109 - loss: 3.6925 - val_accuracy: 0.0875 - val_loss: 24.9526
Learning Rate: 0.001, Batch Size: 64, Test Accuracy: 0.10000000149011612
40/40 ━━━━━━━━━━━━━━━━━━━━ 57s 147ms/step - accuracy: 0.0984 - loss: 6.3711 - val_accuracy: 0.1250 - val_loss: 2.3015
Learning Rate: 0.01, Batch Size: 16, Test Accuracy: 0.11500000208616257
20/20 ━━━━━━━━━━━━━━━━━━━━ 55s 259ms/step - accuracy: 0.1094 - loss: 12.9962 - val_accuracy: 0.1000 - val_loss: 33413056657461608448.0000
Learning Rate: 0.01, Batch Size: 32, Test Accuracy: 0.085

## RandomSearch

In [16]:
from scipy.stats import uniform
from sklearn.model_selection import train_test_split, ParameterSampler
param_dist={
    "learning_rate": uniform(0.001, 0.1),
    "batch_size":[16,32,64]
}
best_accuracy=0
best_params={}
n_iter=5
for params in ParameterSampler(param_dist, n_iter=n_iter, random_state=42):
  model=create_model(learning_rate=params["learning_rate"])
  model.fit(X_train, y_train, epochs=10, batch_size=params["batch_size"], validation_split=0.2, verbose=0)
  loss, accuracy=model.evaluate(X_test, y_test, verbose=0)
  print(f"Params: {params}, Test Accuracy: {accuracy}")
  if accuracy>best_accuracy:
    best_accuracy=accuracy
    best_params=params
print("Best Parameters:", best_params)
print("Best Test Accuracy:", best_accuracy)

Params: {'batch_size': 64, 'learning_rate': np.float64(0.08065429868602329)}, Test Accuracy: 0.07500000298023224
Params: {'batch_size': 64, 'learning_rate': np.float64(0.07419939418114051)}, Test Accuracy: 0.12999999523162842
Params: {'batch_size': 16, 'learning_rate': np.float64(0.0606850157946487)}, Test Accuracy: 0.11500000208616257
Params: {'batch_size': 32, 'learning_rate': np.float64(0.016599452033620267)}, Test Accuracy: 0.07500000298023224
Params: {'batch_size': 64, 'learning_rate': np.float64(0.04692488919658672)}, Test Accuracy: 0.07500000298023224
Best Parameters: {'batch_size': 64, 'learning_rate': np.float64(0.07419939418114051)}
Best Test Accuracy: 0.12999999523162842


한국어 챗봇을 만드세요.
	1. 문장을 넣으면 다음 단어를 생성하는 모델을 만드세요.
	2. 그 모델을 반복적으로 사용하여 완전한 문장을 답하세요.
	3. FastAPI로 감싸서 웹에서 사용할 수 있도록 하세요.

In [7]:
import math
import re
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:
block_size   = 64
n_embd       = 384
n_head       = 6
n_layer      = 6
batch_size   = 64
steps        = 10_000
lr_max       = 3e-4
lr_min       = 3e-5
warmup_steps = 400
dropout      = 0.1
grad_clip    = 1.0

In [9]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
torch.manual_seed(1337)

In [10]:
!curl -o input.txt https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10.2M  100 10.2M    0     0  11.0M      0 --:--:-- --:--:-- --:--:-- 11.0M


In [11]:
text = open("input.txt", encoding="utf-8").read()
text = text.replace("<unk>", "").replace(" @-@ ", "-")

In [12]:
def tokenize(t):
    return re.findall(r"[a-zA-Z0-9']+|[^\s a-zA-Z0-9']", t)

In [13]:
try:
    raw_text = open("input.txt", encoding="utf-8").read()
except FileNotFoundError:
    sys.exit(
        "[ERROR] input.txt not found.\n"
    )

In [14]:
all_tokens = tokenize(text)
vocab      = sorted(set(all_tokens))
vocab_size = len(vocab)
stoi       = {w: i for i, w in enumerate(vocab)}
itos       = {i: w for i, w in enumerate(vocab)}

In [15]:
encode = lambda s: [stoi[t] for t in tokenize(s) if t in stoi]

NO_SPACE_BEFORE = {".", ",", "!", "?", ":", ";", ")", "]", "'s", "'t", "'re", "'ve", "'ll", "'d"}
NO_SPACE_AFTER  = {"(", "["}

In [16]:
data    = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]

In [17]:
def decode(ids):
    words, result = [itos[i] for i in ids], ""
    for i, w in enumerate(words):
        if i == 0 or w in NO_SPACE_BEFORE or (i > 0 and words[i-1] in NO_SPACE_AFTER):
            result += w
        else:
            result += " " + w
    return result

In [18]:
def get_batch(split: str = "train"):
    d  = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x  = torch.stack([d[i    : i + block_size    ] for i in ix])
    y  = torch.stack([d[i + 1: i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

In [19]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv  = nn.Linear(n_embd, 3*n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd,    bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        hd      = C // n_head
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def sh(t): return t.view(B, T, n_head, hd).transpose(1, 2)
        q, k, v = sh(q), sh(k), sh(v)
        att  = q @ k.transpose(-2, -1) / hd**0.5
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        att  = att.masked_fill(~mask, float("-inf"))
        att  = self.drop(F.softmax(att, dim=-1))
        out  = (att @ v).transpose(1, 2).reshape(B, T, C)
        return self.drop(self.proj(out))

In [20]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention()
        self.ln2  = nn.LayerNorm(n_embd)
        self.mlp  = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd, bias=False), nn.GELU(),
            nn.Linear(4*n_embd, n_embd, bias=False), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [21]:
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.Sequential(*[Block() for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)
        self.head    = nn.Linear(n_embd, vocab_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if "proj" in name or "head" in name:
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * n_layer))
            elif p.dim() >= 2:
                nn.init.normal_(p, 0.0, 0.02)

    def forward(self, idx, targets=None):
        T      = idx.size(1)
        x      = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        x      = self.ln_f(self.blocks(x))
        logits = self.head(x)
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50, top_p=0.92):
        EOS = {".", "!", "?"}
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits    = logits[:, -1, :] / temperature

            # top-k
            if top_k > 0:
                v = torch.topk(logits, min(top_k, logits.size(-1))).values
                logits = logits.masked_fill(logits < v[:, [-1]], float("-inf"))

            # top-p (nucleus)
            if top_p < 1.0:
                sl, si = torch.sort(logits, descending=True)
                cum    = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                sl[cum - F.softmax(sl, dim=-1) > top_p] = float("-inf")
                logits = torch.zeros_like(logits).scatter(1, si, sl)

            next_id = torch.multinomial(F.softmax(logits, dim=-1), 1)
            idx     = torch.cat([idx, next_id], dim=1)
            if itos.get(next_id.item(), "") in EOS:
                break
        return idx

In [22]:
@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {s: sum(model(*get_batch(s))[1].item() for _ in range(20)) / 20
           for s in ("train", "val")}
    model.train()
    return out

In [23]:
def get_lr(step):
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    t = (step - warmup_steps) / max(1, steps - warmup_steps)
    return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * t))

In [24]:
def train(model):
    print(f"vocab={vocab_size}  params={sum(p.numel() for p in model.parameters()):,}  device={device}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr_max, betas=(0.9, 0.95), weight_decay=0.1)

    for step in range(steps):
        for pg in opt.param_groups:
            pg["lr"] = get_lr(step)

        x, y = get_batch()
        _, loss = model(x, y)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        opt.step()

        if step % 1000 == 0 or step == steps - 1:
            est = estimate_loss(model)
            print(f"step {step:5d}  lr {get_lr(step):.1e}  train {est['train']:.3f}  val {est['val']:.3f}", flush=True)

    torch.save(model.state_dict(), "mini_gpt.pt")
    print("saved → mini_gpt.pt")

    # 훈련 후 샘플 생성
    model.eval()
    prompt = torch.zeros((1, 1), dtype=torch.long, device=device)
    print("\n--- sample ---")
    print(decode(model.generate(prompt, 200)[0].tolist()))

In [25]:
def chat(model):
    model.load_state_dict(torch.load("mini_gpt.pt", map_location=device))
    model.eval()
    print("Type a prompt and the model will continue it (empty line to quit).")

    while True:
        try:
            prompt = input("> ")
        except EOFError:
            break
        if not prompt:
            break

        ids = [stoi[c] for c in tokenize(prompt) if c in stoi]
        if not ids:
            print("(no tokens from the prompt are in the vocabulary)")
            continue

        idx = torch.tensor([ids], dtype=torch.long, device=device)
        out = model.generate(idx, 200)[0].tolist()
        print(decode(out[len(ids):]))   # 입력 제외, 생성 부분만 출력

In [26]:
model = MiniGPT().to(device)
train(model)

vocab=33254  params=36,190,464  device=cuda
step     0  lr 0.0e+00  train 10.417  val 10.417
step  1000  lr 3.0e-04  train 5.425  val 6.103
step  2000  lr 2.8e-04  train 4.854  val 5.850
step  3000  lr 2.5e-04  train 4.467  val 5.743
step  4000  lr 2.2e-04  train 4.172  val 5.719
step  5000  lr 1.7e-04  train 3.926  val 5.724
step  6000  lr 1.3e-04  train 3.714  val 5.757
step  7000  lr 9.0e-05  train 3.568  val 5.790
step  8000  lr 5.8e-05  train 3.419  val 5.799
step  9000  lr 3.7e-05  train 3.337  val 5.834
step  9999  lr 3.0e-05  train 3.313  val 5.862
saved → mini_gpt.pt

--- sample ---
! " ".


In [ ]:
chat(model)

NameError: name 'model' is not defined

FASTAPI

In [27]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import threading, uvicorn, nest_asyncio

nest_asyncio.apply()

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class GenerateRequest(BaseModel):
    text: str
    max_new_tokens: int = 60
    temperature: float = 0.8
    top_k: int = 50
    top_p: float = 0.92

class GenerateResponse(BaseModel):
    prompt: str
    generated: str
    full_text: str

@app.post("/generate", response_model=GenerateResponse)
def generate(req: GenerateRequest):
    try:
        ids = encode(req.text)
        if not ids:
            raise HTTPException(status_code=400, detail="No known tokens.")
        idx = torch.tensor([ids], dtype=torch.long, device=device)
        out_ids = model.generate(idx, req.max_new_tokens)[0].tolist()
        return GenerateResponse(
            prompt=req.text,
            generated=decode(out_ids[len(ids):]),
            full_text=decode(out_ids),
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/", response_class=HTMLResponse)
def index():
    return """
    <h2>MiniGPT</h2>
    <textarea id="p" rows="3" cols="50"></textarea><br>
    <button onclick="gen()">Generate</button>
    <p id="status"></p>
    <pre id="out"></pre>
    <script>
    async function gen() {
        const text = document.getElementById('p').value;
        document.getElementById('status').textContent = '생성 중...';
        try {
            const res = await fetch('/generate', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify({text: text, max_new_tokens: 60}),
                credentials: 'include'
            });
            const d = await res.json();
            document.getElementById('out').textContent = d.full_text || d.detail;
            document.getElementById('status').textContent = '완료';
        } catch(e) {
            document.getElementById('status').textContent = '에러: ' + e;
        }
    }
    </script>
    """
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()

import time; time.sleep(2)
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))

INFO:     Started server process [306]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


https://8000-gpu-a100-s-kkb-usc1f2-1khsukivgmloh-f.us-central1-2.prod.colab.dev
